# Reading the challenge dataset

Two ways in: straight to `pandas` for bulk work, or through `mlcroissant` when you
want the metadata such as field descriptions, units, which turbine belongs to which
split.

| File | Rows | Turbines | Labels |
|---|---|---|---|
| `train.parquet` | 17.2 M | PPP_WTG12, PPP_WTG13, PPP_WTG14 | yes, where WindFit measured |
| `validate.parquet` | 5.7 M | PPP_WTG17 | no, used for public leaderboard |
| `test.parquet` | 4.7 M | SSS_WTG06 | no, used in the final round |
| `context.parquet` | 59.1 M | 11 neighbours | no |
| `turbine_locations_PPP.csv` | 11 | — | layout |
| `turbine_locations_SSS.csv` | 5 | — | layout |

**IMPORTANT:** 
1. High-res values are written **only when they
change**, so a null means "unchanged since the last update". 
Forward fill it, don't drop it. 
2.  `train.parquet` covers every day of the two years, including
days with no measurement, so `yaw_misalignment_deg` is null on most rows.

In [21]:
import itertools
import json
from pathlib import Path

import pandas as pd
from mlcroissant import Dataset
from rdflib import Graph, URIRef

METADATA_FILE = "metadata.croissant.json"
DATA_DIR = Path("data")

## Helpers

In [22]:
def records_to_df(records_iter, head: int | None = None) -> pd.DataFrame:
    """Collect an mlcroissant records() iterator into a DataFrame."""
    rows = list(itertools.islice(records_iter, head)) if head else list(records_iter)
    return pd.DataFrame(rows)


def bare_names(df: pd.DataFrame) -> pd.DataFrame:
    """Strip the record-set prefix: 'train/WindSpeed' -> 'WindSpeed'.

    mlcroissant also hands back text columns as bytes, so decode those too.
    """
    df = df.rename(columns={c: c.split("/")[-1] for c in df.columns})
    for col in df.columns:
        if df[col].map(lambda v: isinstance(v, bytes)).any():
            df[col] = df[col].str.decode("utf-8")
    return df

## Straight to pandas

Nothing here needs `mlcroissant`. `context.parquet` is 59 M rows, so select the
columns you need rather than loading it whole.

In [23]:
train = pd.read_parquet(DATA_DIR / "train.parquet")
print(train.shape, "|", train.turbine_id.unique())
print(f"labelled rows: {train.yaw_misalignment_deg.notna().mean():.1%}")
train.head(10)

(17159518, 10) | <ArrowStringArray>
['PPP_WTG12', 'PPP_WTG13', 'PPP_WTG14']
Length: 3, dtype: str
labelled rows: 32.4%


,turbine_id,ts,GenSpeed,NacDir,WindSpeed,WindDir,PitchAngle,RotSpeed,Power,yaw_misalignment_deg
0,PPP_WTG12,2023-01-01 00:00:09,2024.40,NaN,NaN,176.4,-2.739,NaN,NaN,NaN
1,PPP_WTG12,2023-01-01 00:00:10,NaN,NaN,8.19,NaN,NaN,19.32,2825.55,NaN
2,PPP_WTG12,2023-01-01 00:00:26,2012.52,NaN,11.70,173.7,-0.693,19.20,3157.80,NaN
3,PPP_WTG12,2023-01-01 00:00:38,2046.72,NaN,12.87,174.6,-2.189,19.56,3643.20,NaN
4,PPP_WTG12,2023-01-01 00:00:46,NaN,NaN,NaN,175.0,NaN,NaN,NaN,NaN
5,PPP_WTG12,2023-01-01 00:00:50,2034.72,NaN,12.51,NaN,-0.957,19.44,3248.40,NaN
6,PPP_WTG12,2023-01-01 00:01:11,2006.04,NaN,14.31,175.8,0.099,19.08,3570.30,NaN
7,PPP_WTG12,2023-01-01 00:01:27,2014.08,NaN,11.70,176.8,-0.055,19.20,3372.90,NaN
8,PPP_WTG12,2023-01-01 00:01:39,1986.60,NaN,8.10,178.5,-1.045,18.96,2498.25,NaN
9,PPP_WTG12,2023-01-01 00:01:59,1981.08,NaN,9.00,183.3,-0.990,18.84,1767.75,NaN


In [24]:
# One label per turbine-day, so collapse to days to see the actual ground truth
labels = (train.dropna(subset=["yaw_misalignment_deg"])
          .groupby(["turbine_id", train.ts.dt.normalize().rename("date")])
          .yaw_misalignment_deg.first().reset_index())
print(f"{len(labels)} labelled turbine-days")
labels.groupby("turbine_id").yaw_misalignment_deg.agg(["size", "mean", "min", "max"]).round(2)

702 labelled turbine-days


,size,mean,min,max
turbine_id,,,,
PPP_WTG12,195,-2.28,-3.20,-1.40
PPP_WTG13,312,-7.46,-11.60,-3.05
PPP_WTG14,195,-1.70,-2.59,-0.88


In [25]:
validate = pd.read_parquet(DATA_DIR / "validate.parquet")   # public leaderboard turbine
test = pd.read_parquet(DATA_DIR / "test.parquet")           # final round, labels withheld
print("validate", validate.shape, "| test", test.shape)

validate (5732985, 9) | test (4665730, 9)


In [26]:
# 59 M rows: pick columns, and filter to the turbines you actually want
context = pd.read_parquet(
    DATA_DIR / "context.parquet",
    columns=["turbine_id", "ts", "WindSpeed", "Power", "NacDir", "WindDir"],
)
print(context.shape, "|", sorted(context.turbine_id.unique()))

(59113094, 6) | ['PPP_WTG07', 'PPP_WTG08', 'PPP_WTG11', 'PPP_WTG15', 'PPP_WTG16', 'PPP_WTG18', 'PPP_WTG33', 'SSS_WTG04', 'SSS_WTG05', 'SSS_WTG07', 'SSS_WTG16']


### Reconstructing the signals

Forward fill first: nulls mean "unchanged", not "missing". Vane angle is the
signal most methods want and it is not shipped: derive it from wind direction and
nacelle position, wrapped to ±180°.

In [27]:
def prepare(df: pd.DataFrame) -> pd.DataFrame:
    """Forward fill per turbine, then add the vane angle."""
    signals = ["GenSpeed", "NacDir", "WindSpeed", "WindDir", "PitchAngle", "RotSpeed", "Power"]
    cols = [c for c in signals if c in df.columns]
    out = df.sort_values(["turbine_id", "ts"]).copy()
    out[cols] = out.groupby("turbine_id")[cols].ffill()
    out["vane_deg"] = (out.WindDir - out.NacDir + 180) % 360 - 180
    return out


sample = prepare(train[train.turbine_id == "PPP_WTG13"].head(200_000))
print("nulls before ffill:", train.head(200_000).NacDir.isna().mean().round(3),
      "| after:", sample.NacDir.isna().mean().round(3))
sample[["ts", "WindSpeed", "Power", "vane_deg", "yaw_misalignment_deg"]].head(3)

nulls before ffill: 0.927 | after: 0.0


,ts,WindSpeed,Power,vane_deg,yaw_misalignment_deg
5740015,2023-01-01 00:00:09,11.07,2625.9,NaN,-11.598047
5740016,2023-01-01 00:00:10,11.07,2625.9,NaN,-11.598047
5740017,2023-01-01 00:00:26,9.09,2431.2,NaN,-11.598047


## Turbine layout

One file per site, because the coordinates are relative to a different origin at
each.
`x_m` is easting, `y_m` is northing.

In [28]:
locations = pd.concat([
    pd.read_csv(DATA_DIR / f"turbine_locations_{site}.csv").assign(site=site)
    for site in ("PPP", "SSS")])
locations

,turbine_id,role,x_m,y_m,site
0,PPP_WTG07,context,-754.6,1534.2,PPP
1,PPP_WTG08,context,-466.5,1476.0,PPP
2,PPP_WTG11,context,-125.7,-326.3,PPP
3,PPP_WTG12,train,0.0,0.0,PPP
4,PPP_WTG13,train,235.3,473.1,PPP
5,PPP_WTG14,train,343.1,772.2,PPP
6,PPP_WTG15,context,2583.3,1934.4,PPP
7,PPP_WTG16,context,2639.6,2203.1,PPP
8,PPP_WTG17,validate,2830.6,2476.7,PPP
9,PPP_WTG18,context,3121.7,2857.3,PPP


In [9]:
import numpy as np

def neighbours(turbine_id, k=4):
    """Closest turbines at the same site, with distance and bearing."""
    site = locations.loc[locations.turbine_id == turbine_id, "site"].iloc[0]
    here = locations[locations.site == site].set_index("turbine_id")
    origin = here.loc[turbine_id]
    dx, dy = here.x_m - origin.x_m, here.y_m - origin.y_m
    out = pd.DataFrame({
        "distance_m": np.hypot(dx, dy).round(0),
        "bearing_deg": (np.degrees(np.arctan2(dx, dy)) % 360).round(1),  # clockwise from north
        "role": here.role,
    }).drop(index=turbine_id).sort_values("distance_m")
    return out.head(k)


neighbours("PPP_WTG17")

,distance_m,bearing_deg,role
turbine_id,,,
PPP_WTG16,334.0,214.9,context
PPP_WTG18,479.0,37.4,context
PPP_WTG15,596.0,204.5,context
PPP_WTG33,678.0,57.4,context


## Through mlcroissant

The Croissant sidecar carries a description and a unit for every field. Record
sets stream row by row through Python, so it is the wrong tool for bulk reads. Always pass `head=N` on the parquet-backed sets and use `pd.read_parquet` for real
work. The location sets are small enough to read whole.

In [10]:
ds = Dataset(jsonld=METADATA_FILE)
[rs.uuid for rs in ds.metadata.record_sets]

['train',
 'validate',
 'test',
 'context',
 'turbine_locations_ppp',
 'turbine_locations_sss']

In [11]:
bare_names(records_to_df(ds.records("turbine_locations_PPP".lower())))

,turbine_id,role,x_m,y_m
0,PPP_WTG07,context,-754.6,1534.2
1,PPP_WTG08,context,-466.5,1476.0
2,PPP_WTG11,context,-125.7,-326.3
3,PPP_WTG12,train,0.0,0.0
4,PPP_WTG13,train,235.3,473.1
5,PPP_WTG14,train,343.1,772.2
6,PPP_WTG15,context,2583.3,1934.4
7,PPP_WTG16,context,2639.6,2203.1
8,PPP_WTG17,validate,2830.6,2476.7
9,PPP_WTG18,context,3121.7,2857.3


In [12]:
# parquet-backed sets: always cap with head
bare_names(records_to_df(ds.records("train"), head=5))

,turbine_id,ts,GenSpeed,NacDir,WindSpeed,WindDir,PitchAngle,RotSpeed,Power,yaw_misalignment_deg
0,PPP_WTG12,2023-01-01 00:00:09,2024.40,None,NaN,176.4,-2.739,NaN,NaN,None
1,PPP_WTG12,2023-01-01 00:00:10,NaN,None,8.19,NaN,NaN,19.32,2825.55,None
2,PPP_WTG12,2023-01-01 00:00:26,2012.52,None,11.70,173.7,-0.693,19.20,3157.80,None
3,PPP_WTG12,2023-01-01 00:00:38,2046.72,None,12.87,174.6,-2.189,19.56,3643.20,None
4,PPP_WTG12,2023-01-01 00:00:46,NaN,None,NaN,175.0,NaN,NaN,NaN,None


## Field metadata

Every field carries a description and, where meaningful, a unit IRI from
[QUDT](https://qudt.org/). Resolving those IRIs against the QUDT vocabulary gives
you readable labels and symbols.

In [13]:
# mlcroissant keys Field.jsonld by rdflib URIRef, not by the short name used in the
# JSON file -- "unit" is unmapped in the @context so it resolves against @vocab
UNIT_KEY = URIRef("https://schema.org/unit")
RDFS_LABEL = URIRef("http://www.w3.org/2000/01/rdf-schema#label")
QUDT_SYMBOL = URIRef("http://qudt.org/schema/qudt/symbol")
QUDT_UNITS = "https://qudt.org/2.1/vocab/unit.ttl"


def english(graph, subject, predicate) -> str:
    """First English label, falling back to any. QUDT labels are multilingual."""
    values = list(graph.objects(subject, predicate))
    for v in values:
        if getattr(v, "language", None) == "en":
            return str(v)
    return str(values[0]) if values else ""


def field_unit(f) -> str:
    """Unit IRI for a field, or '' if it has none."""
    raw = (getattr(f, "jsonld", None) or {}).get(UNIT_KEY)
    if not raw:
        return ""
    return str(raw) if not isinstance(raw, dict) else str(raw.get("@id", ""))


def record_set_schema(record_set_id: str, metadata, units_graph: Graph | None = None) -> pd.DataFrame:
    """One row per field: name, dataType, unit and description.

    Pass a units graph to resolve the unit IRIs to readable labels and symbols.
    """
    rs = next(r for r in metadata.record_sets if r.uuid == record_set_id)
    rows = []
    for f in rs.fields:
        unit = field_unit(f)
        row = dict(field=f.uuid.split("/")[-1],
                   dataType=getattr(f.data_type, "__name__", str(f.data_type)),
                   unit=unit.rsplit("/", 1)[-1] if unit else "",
                   description=f.description)
        if units_graph is not None:
            u = URIRef(unit) if unit else None
            row["unit_label"] = english(units_graph, u, RDFS_LABEL) if u else ""
            row["unit_symbol"] = english(units_graph, u, QUDT_SYMBOL) if u else ""
        row["unit_iri"] = unit
        rows.append(row)
    return pd.DataFrame(rows)


schema = record_set_schema("train", ds.metadata)
print(f"{(schema.unit != '').sum()} of {len(schema)} fields carry a unit")
schema[["field", "dataType", "unit"]]

8 of 10 fields carry a unit


,field,dataType,unit
0,turbine_id,bytes,
1,ts,Timestamp,
2,GenSpeed,float,REV-PER-MIN
3,NacDir,float,DEG
4,WindSpeed,float,M-PER-SEC
5,WindDir,float,DEG
6,PitchAngle,float,DEG
7,RotSpeed,float,REV-PER-MIN
8,Power,float,KiloW
9,yaw_misalignment_deg,float,DEG


In [14]:
# Resolve the unit IRIs to readable labels and symbols. One network fetch.
try:
    units_graph = Graph()
    units_graph.parse(QUDT_UNITS, format="turtle")
    resolved = record_set_schema("train", ds.metadata, units_graph)
    display(resolved[["field", "unit", "unit_label", "unit_symbol"]])
except Exception as exc:
    print(f"[skipped, no network] {exc}")

,field,unit,unit_label,unit_symbol
0,turbine_id,,,
1,ts,,,
2,GenSpeed,REV-PER-MIN,Revolution per Minute,rev/min
3,NacDir,DEG,Degree,°
4,WindSpeed,M-PER-SEC,metre per second,m/s
5,WindDir,DEG,Degree,°
6,PitchAngle,DEG,Degree,°
7,RotSpeed,REV-PER-MIN,Revolution per Minute,rev/min
8,Power,KiloW,Kilowatt,kW
9,yaw_misalignment_deg,DEG,Degree,°


In [15]:
# Units are on the layout fields too
record_set_schema("turbine_locations_ppp", ds.metadata)[["field", "dataType", "unit"]]

,field,dataType,unit
0,turbine_id,bytes,
1,role,bytes,
2,x_m,float,M
3,y_m,float,M


## Where to go next

`train.parquet` gives you three turbines at roughly −1.7°, −2.3° and −7.5°, all
within 800 m of each other and measured over the same months. That contrast is the
most useful thing in the dataset: two well-aligned machines and one badly
misaligned one, under the same weather.

Bear in mind the labels are a smoothed daily estimate. Consecutive days barely
move, so 702 labelled days hold only three independent misalignment levels. Treat
your effective sample size as 3, not 702, when deciding how much a validation
score is telling you.